# Validation Test Evaluation - Rising Bubble Benchmark Testcase 2

This notebook and the corresponding simulation setup notebook (RisingBubble_Testcase2_Run.ipynb) are part of published results (cf. 7.3.2) found in *On a marching level-set method for extended discontinuous Galerkin methods for incompressible two-phase flows: Application to two-dimensional settings* (https://doi.org/10.1002%2Fnme.6853). We compare our results against the rising bubble benchmark test case established by Hysing et al ( https://doi.org/10.1002/fld.1934)

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
wmg.Init("XNSEpaper_RisingBubble");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\XNSEpaper_RisingBubble");

Opening existing database '\\fdygitrunner\ValidationTests\databases\XNSEpaper_RisingBubble'.


In [3]:
using System.IO;
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
var sessions = wmg.Sessions;
//var sessions = databases.Pick(0).Sessions;
sessions

#0: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh40_Restart1	11/27/2025 10:42:39 AM	5bed5ae8...
#1: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh20_Restart1	11/27/2025 2:53:28 PM	2a843101...
#2: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh40_Restart1*	11/27/2025 2:53:36 PM	7f2a9aa3...
#3: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh40*	11/11/2025 2:37:08 PM	eaddac04...
#4: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh20*	11/11/2025 2:36:53 PM	3766e111...
#5: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh40*	11/11/2025 2:36:43 PM	a7c228ba...
#6: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k2_mesh60	11/5/2025 11:44:04 AM	89908644...
#7: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k2_mesh40	11/5/2025 11:43:54 AM	f3f34e8d...
#8: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh20	11/5/2025 11:44:24 AM	180473a4...
#9: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k2_mesh20	11/5/2025 11:43:44 AM	

In [6]:
// foreach (var s in sessions) {
//     if(s.Name.Contains("RisingBubble") && s.CreationTime.Date == new DateTime(2025, 11, 5) && !s.SuccessfulTermination) {
//         //s.Delete(true);
//         Console.WriteLine($"Deleted session {s.Name} from {s.CreationTime.Date}");
//     }
// }

In [ ]:
// var sessions = wmg.Sessions;
// sessions

# Test case 2 - Dimpled cap with filaments

In [5]:
sessions

#0: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh40_Restart1	11/27/2025 10:42:39 AM	5bed5ae8...
#1: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh20_Restart1	11/27/2025 2:53:28 PM	2a843101...
#2: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh40_Restart1*	11/27/2025 2:53:36 PM	7f2a9aa3...
#3: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh40*	11/11/2025 2:37:08 PM	eaddac04...
#4: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh20*	11/11/2025 2:36:53 PM	3766e111...
#5: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh40*	11/11/2025 2:36:43 PM	a7c228ba...
#6: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k2_mesh60	11/5/2025 11:44:04 AM	89908644...
#7: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k2_mesh40	11/5/2025 11:43:54 AM	f3f34e8d...
#8: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh20	11/5/2025 11:44:24 AM	180473a4...
#9: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k2_mesh20	11/5/2025 11:43:44 AM	

In [6]:
int[] Resolutions = { 20, 40, 80 };

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct14_000000.XNSEpaper_RisingBubble");

In [7]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
foreach (int Res in Resolutions) {
    string studyName = $"RisingBubble_tc2_ConvStudy_k2_mesh{Res}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName));
    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found!");
        
        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.ToLower().EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }

    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.ToLower().EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
evalSess

Searching for: RisingBubble_tc2_ConvStudy_k2_mesh20
Restart session found! Take last run
Searching for: RisingBubble_tc2_ConvStudy_k2_mesh40
Restart session found! Take last run
Searching for: RisingBubble_tc2_ConvStudy_k2_mesh80
Not found!


#0: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh20_Restart1	11/27/2025 2:53:28 PM	2a843101...
#1: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh40_Restart1*	11/27/2025 2:53:36 PM	7f2a9aa3...


In [17]:
NUnit.Framework.Assert.AreEqual(3, evalSess.Count, $"Not enough sessions for evaluation.");

### compute times

In [50]:
foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = nameData.Length;
    for (int i = 0; i < nameLength; i++) {
        if (nameData[i].StartsWith("restart"))
            continue;
        if (i == 0)
            sessName += nameData[i];
        else    
            sessName += "_" + nameData[i];
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    var currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
    int timesteps = currTimesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currTimesteps.Last().PhysicalTime;
    TimeSpan computeTime = currTimesteps.Last().CreationTime - currSI.Timesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        var rSI = rSIs.Single();
        procSIs.Push(rSI);
        currSI = rSI;
        currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
        computeTime = computeTime + (currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime);
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps} (t_end = {physicalTime}); compute time = {computeTime.Days}:{computeTime.Hours}");
}

Session - RisingBubble_tc2_ConvStudy_k2_mesh40_Restart1: timesteps = 636 (t_end = 1.904999999999968); compute time = 0:18
Session - RisingBubble_tc2_ConvStudy_k2_mesh20_Restart1: timesteps = 334 (t_end = 3.005999999999978); compute time = 0:4


### read log data

In [ ]:
evalSess = evalSess.OrderBy(s => s.KeysAndQueries["Grid:hMax"]).ToList();

In [12]:
static List<Plot2Ddata> evalData = new List<Plot2Ddata>();

In [13]:
Plot2Ddata riseHeightData = evalSess.ReadLogDataValue("BenchmarkQuantities_RisingBubble", "center of mass - y");
evalData.Add(riseHeightData);

Processing session: RisingBubble_tc2_ConvStudy_k2_mesh40_Restart1
  Found restart session: eaddac04-37e0-4772-8e3e-07031f7a22d0
  Merging data from 2 sessions.
... Done
Processing session: RisingBubble_tc2_ConvStudy_k2_mesh20_Restart1
  Found restart session: 3766e111-2bfd-41d2-85fe-d19b0a227ba9
  Merging data from 2 sessions.
... Done
Build DataSet


In [14]:
Plot2Ddata circData = evalSess.ReadLogDataValue("BenchmarkQuantities_RisingBubble", "circularity");
evalData.Add(circData);

Processing session: RisingBubble_tc2_ConvStudy_k2_mesh40_Restart1
  Found restart session: eaddac04-37e0-4772-8e3e-07031f7a22d0
  Merging data from 2 sessions.
... Done
Processing session: RisingBubble_tc2_ConvStudy_k2_mesh20_Restart1
  Found restart session: 3766e111-2bfd-41d2-85fe-d19b0a227ba9
  Merging data from 2 sessions.
... Done
Build DataSet


In [15]:
Plot2Ddata riseVelocityData = evalSess.ReadLogDataValue("BenchmarkQuantities_RisingBubble", "rise velocity", new string[] { "mean velocity y" } );
evalData.Add(riseVelocityData);

Processing session: RisingBubble_tc2_ConvStudy_k2_mesh40_Restart1
  Found restart session: eaddac04-37e0-4772-8e3e-07031f7a22d0
  Merging data from 2 sessions.
... Done
Processing session: RisingBubble_tc2_ConvStudy_k2_mesh20_Restart1
  Found restart session: 3766e111-2bfd-41d2-85fe-d19b0a227ba9
  Merging data from 2 sessions.
... Done
Build DataSet


In [16]:
string caseName = "testcase2";

In [17]:
static string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [18]:
// g1 TU Dortmund (TP2D); g2 EPFL Lausanne (FreeLIFE); g3 Uni Magdebug (MooNMD)
static string[] groups = new string[] {"TP2D", "FreeLIFE", "MooNMD"};
static int[] datLvl;
static string datCase;
if(caseName.Equals("testcase1")) {
    datCase = "c1g";
    datLvl  = new int[] {7, 3, 4};    // testcase 1
} 
if(caseName.Equals("testcase2")) {     
    datCase = "c2g";
    datLvl  = new int[] {8, 3, 4};    // testcase 2
}

Plot2Ddata[,] dataRef = new Plot2Ddata[4,3];
for (int grp = 1; grp <= groups.Length; grp++) {
    Plot2Ddata[] datSet = new Plot2Ddata[4];
    // 1: area 2: circularity 3: center y 4: rise velocity
    for (int j = 0; j < 4; j++) {
        datSet[j] = new Plot2Ddata();
    }

    string dat  = datCase+grp+"l"+datLvl[grp - 1]+".txt";
    string path = (userName.Equals(@"FDY\jenkinsci")) ? dat : @"Featflow_referenceData\" + dat;
        string[] lines = File.ReadAllLines(path);
        double[] time = new double[lines.Length];
        double[][] valueDat = new double[4][];
        for(int j = 0; j < 4; j++)
            valueDat[j] = new double[lines.Length];

        for (int i = 0; i < lines.Length; i++) {
            //var datString = lines[i].Split(new string[] {" "}, StringSplitOptions.RemoveEmptyEntries);
            //Console.WriteLine("num split strings at 0: {0}", datString[0]);
            time[i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[0]);            
            for (int j = 0; j < 4; j++) {
                valueDat[j][i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[j+1]);
            }
        }        
        // Build DataSet
        for (int j = 0; j < 4; j++) {
            string datName = groups[grp-1]+"-l"+datLvl[grp - 1];
            datSet[j] = (new Plot2Ddata(new KeyValuePair<string, double[][]>(datName, new double[][] { time, valueDat[j] })));
        }
    
    for (int j = 0; j < 4; j++) {
        dataRef[j,grp-1] = datSet[j];
    }
}

## FIGURE 10

In [19]:
public static (double[] x, double[] y) GetTerminalShapeInterfacePoints(ISessionInfo sess) {

    var terminalStep = sess.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber).Last();
    DGField phi = terminalStep.GetField("Phi");
    LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet"); 
    LevSet.Acc(1.0, phi);           
    LevelSetTracker LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
    LsTrk.UpdateTracker(terminalStep.PhysicalTime);

    MultidimensionalArray interP = BoSSS.Solution.XNSECommon.XNSEUtils.GetInterfacePoints(LsTrk, LevSet, quadRuleOrderForNodeSet:10);
    double[] x = interP.ExtractSubArrayShallow(new int[] { -1, 0 }).To1DArray();
    double[] y = interP.ExtractSubArrayShallow(new int[] { -1, 1 }).To1DArray();

    return (x, y);
}

In [20]:
public static Plot2Ddata GetShapeComparison_Plot2Ddata(List<ISessionInfo> dataSess, string caseName, string[] study) {

    Plot2Ddata datShape = new Plot2Ddata();

    int lcIdx = 1;
    for(int i = 0; i < study.Length; i++) {
        foreach (var dataS in dataSess) {
            if (dataS.Name.Contains(study[i])) {
                var shape = GetTerminalShapeInterfacePoints(dataS);

                var fmt = new PlotFormat();
                fmt.Style = Styles.Lines;
                fmt.Style      = Styles.Points;
                fmt.PointType  = PointTypes.Dot;
                fmt.LineColor = (LineColors)(lcIdx);

                datShape.AddDataGroup((study[i]).Replace("_", "-"), shape.x, shape.y, fmt);
                lcIdx++;
            }
        }
    }

    // Add reference shapes
    // g1 TU Dortmund (TP2D); g2 EPFL Lausanne (FreeLIFE); g3 Uni Magdebug (MooNMD)
    string[] groups = new string[] {"TP2D", "FreeLIFE", "MooNMD"};
    int[] datLvl = null;
    string datCase = "";
    if(caseName.Equals("testcase1")) {
        datCase = "c1g";
        datLvl  = new int[] {7, 3, 4};    // testcase 1
    } 
    if(caseName.Equals("testcase2")) {     
        datCase = "c2g";
        datLvl  = new int[] {8, 3, 4};    // testcase 2
    }

    for (int grp = 1; grp <= groups.Length; grp++) {

        string dat  = datCase+grp+"l"+datLvl[grp - 1]+"s.txt";
        string path = (userName.Equals(@"FDY\jenkinsci")) ? dat : @"Featflow_referenceData\" + dat;
        string[] lines = File.ReadAllLines(path);
        double[] x = new double[lines.Length];
        double[] y = new double[lines.Length];

        for (int i = 0; i < lines.Length; i++) {
            x[i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[0]);            
            y[i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[1]);      
        }   

        string datName = groups[grp-1]+"-l"+datLvl[grp - 1]+"s";

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines;
        fmt.Style      = Styles.Points;
        fmt.PointType  = PointTypes.Dot;
        fmt.LineColor = (LineColors)(9 - grp);

        datShape.AddDataGroup(datName, x, y , fmt);
    }

    datShape.LegendAlignment = new string[] { "o", "t", "r" };

    return datShape;
}

In [30]:
var Fig10 = GetShapeComparison_Plot2Ddata(evalSess, caseName, new string[] { "k2_mesh80" });
Fig10.XrangeMin = 0.7;
Fig10.XrangeMax = 0.9;
Fig10.YrangeMin = 0.5;
Fig10.YrangeMax = 1.1;

Fig10.ToGnuplot().PlotSVG(xRes:500,yRes:500)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 0.9 
 
 
 
 
 1 
 
 
 
 
 1.1 
 
 
 
 
 0.7 
 
 
 
 
 0.75 
 
 
 
 
 0.8 
 
 
 
 
 0.85 
 
 
 
 
 0.9 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 TP2D-l8s 
 
 
 TP2D-l8s 
 
 
 
 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

## FIGURE 11 + B2 - Comparison with benchmark quantities

In [ ]:
public static Plot2Ddata GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(List<Plot2Ddata> data, Plot2Ddata[,] refData, string[] qNameHints, string[] LegendAlign, double xMin, double xMax) {

    Plot2Ddata plt =  new Plot2Ddata();

    int index = -1;
    for (int i = 0; i < data.Count(); i++) {
        if (qNameHints.Contains(data.ElementAt(i).Ylabel))
            index = i;
    }
    
    int lcIdx = 1;
    var datGroups = data.ElementAt(index).dataGroups;
    for (int i = 0; i < datGroups.Count(); i++) {
        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        if (nameData[3] == "k2" && nameData[4] == "mesh80"
            || nameData[3] == "k3" && nameData[4] == "mesh60") {

            var fmt = new PlotFormat();
            fmt.Style = Styles.Lines; 
            fmt.DashType = DashTypes.Solid;
            fmt.LineWidth = 2;
            fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

            plt.AddDataGroup($"{nameData[3]}-{nameData[4]}", datGroups[i].Abscissas, datGroups[i].Values, fmt);
            lcIdx++;
        }
    }

    string[] refProps = new string[] { "area", "circularity", "center of mass - y", "rise velocity" };
    index = -1;
    for (int i = 0; i < refProps.Length; i++) {
        if (qNameHints.Contains(refProps[i]))
            index = i;
    }

    for (int i = 0; i < refData.GetLength(1); i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.DotDashed;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx);
        
        // Add reference data
        plt.AddDataGroup(refData[index, i].dataGroups[0], fmt);
        lcIdx++;
    }

    plt.Xlabel = "Time";
    plt.Ylabel = qNameHints[0];

    plt.XrangeMin = xMin;
    plt.XrangeMax = xMax;
    // plt.YrangeMin = yMin;
    // plt.YrangeMax = yMax;

    plt.LegendAlignment = LegendAlign;
    plt.ShowLegend = LegendAlign != null ? true : false;
    
    return plt;
}

In [49]:
Plot2Ddata[,] Fig11 = new Plot2Ddata[3, 2];

Fig11[0, 0] = GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(evalData, dataRef, new string[] {"rise velocity"}, new string[] { "i", "b", "r" }, 0.0, 3.0);
Fig11[0, 1] = GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(evalData, dataRef, new string[] {"rise velocity"}, null, 0.5, 2.2);

Fig11[1, 0] = GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(evalData, dataRef, new string[] {"circularity"}, new string[] { "i", "t", "r" }, 0.0, 3.0);
Fig11[1, 1] = GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(evalData, dataRef, new string[] {"circularity"}, null, 1.5, 3.0);

Fig11[2, 0] = GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(evalData, dataRef, new string[] {"center of mass - y"}, new string[] { "i", "b", "r" }, 0.0, 3.0);
Fig11[2, 1] = GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(evalData, dataRef, new string[] {"center of mass - y"}, null, 2.0, 3.0);

Fig11.ToGnuplot().PlotSVG(xRes:1200,yRes:1200)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.05 
 
 
 
 
 0 
 
 
 
 
 0.05 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rise velocity 
 
 
 
 
 Time 
 
 
 
 
 k2-mesh40 
 
 
 
 
 k2-mesh40 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M349.2,245.4 L402.6,245.4 M86.6,294.2 L87.0,292.5 L87.5,290.9 L87.9,289.3 L88.4,287.7 L88.8,286.1
 L89.3,284.5 L89.8,282.9 L90.2,281.4 L90.7,279.8 L91.1,278.2 L91.6,276.7 L92.1,275.1 L92.5,273.6
 L93.0,272.0 L93.4,270.5 L93.9,268.9 L94.3,267.4 L94.8,265.9 L95.3,264.4 L95.7,262.9 L96.2,261.4
 L96.6,259.9 L97.1,258.4 L97.6,256.9 L98.0,255.4 L98.5,253.9 L98.9,252.4 L99.4,250.9 L99.8,249.5
 L100.3,248.0 L100.8,246.6 L101.2,245.1 L101.7,243.7 L102.1,242.2 L102.6,240.8 L103.0,239.3 L103.5,237.9
 L104.0,236.5 L104.4,235.1 L104.9,233.7 L105.3,232.2 L105.8,230.8 L106.3,229.4 L106.7,228.0 L107.2,226.6
 L107.6,225.3 L108.1,223.9 L108.5,222.5 L109.0,221.1 L109.5,219.8 L109.9,218.4 L110.4,217.0 L110.8,215.7
 L111.3,214.3 L111.7,213.0 L112.2,211.6 L112.7,210.3 L113.1,209.0 L113.6,207.7 L114.0,206.3 L114.5,205.0
 L115.0,203.7 L115.4,202.4 L115.9,201.1 L116.3,199.8 L116.8,198.5 L117.2,197.2 L117.7,195.9 L118.2,194.7
 L118.6,193.4 L119.1,192.1 L119.5,190.9 L120.0,189.6 L120.5,188.4 L120.9,187.1 L121.4,185.9 L121.8,184.6
 L122.3,183.4 L122.7,182.2 L123.2,181.0 L123.7,179.7 L124.1,178.5 L124.6,177.3 L125.0,176.1 L125.5,174.9
 L125.9,173.8 L126.4,172.6 L126.9,171.4 L127.3,170.2 L127.8,169.1 L128.2,167.9 L128.7,166.8 L129.2,165.6
 L129.6,164.5 L130.1,163.3 L130.5,162.2 L131.0,161.1 L131.4,160.0 L131.9,158.9 L132.4,157.8 L132.8,156.7
 L133.3,155.6 L133.7,154.5 L134.2,153.4 L134.6,152.4 L135.1,151.3 L135.6,150.2 L136.0,149.2 L136.5,148.1
 L136.9,147.1 L137.4,146.1 L137.9,145.1 L138.3,144.1 L138.8,143.0 L139.2,142.0 L139.7,141.0 L140.1,140.1
 L140.6,139.1 L141.1,138.1 L141.5,137.2 L142.0,136.2 L142.4,135.3 L142.9,134.3 L143.4,133.4 L143.8,132.5
 L144.3,131.5 L144.7,130.6 L145.2,129.7 L145.6,128.8 L146.1,128.0 L146.6,127.1 L147.0,126.2 L147.5,125.4
 L147.9,124.5 L148.4,123.7 L148.8,122.8 L149.3,122.0 L149.8,121.2 L150.2,120.4 L150.7,119.6 L151.1,118.8
 L151.6,118.0 L152.1,117.2 L152.5,116.5 L153.0,115.7 L153.4,115.0 L153.9,114.2 L154.3,113.5 L154.8,112.8
 L155.3,112.0 L155.7,111.3 L156.2,110.6 L156.6,110.0 L157.1,109.3 L157.5,108.6 L158.0,108.0 L158.5,107.3
 L158.9,106.7 L159.4,106.0 L159.8,105.4 L160.3,104.8 L160.8,104.2 L161.2,103.6 L161.7,103.0 L162.1,102.5
 L162.6,101.9 L163.0,101.3 L163.5,100.8 L164.0,100.2 L164.4,99.7 L164.9,99.2 L165.3,98.7 L165.8,98.2
 L166.3,97.7 L166.7,97.2 L167.2,96.7 L167.6,96.2 L168.1,95.8 L168.5,95.3 L169.0,94.9 L169.5,94.5
 L169.9,94.1 L170.4,93.6 L170.8,93.2 L171.3,92.8 L171.7,92.4 L172.2,92.1 L172.7,91.7 L173.1,91.3
 L173.6,91.0 L174.0,90.7 L174.5,90.3 L175.0,90.0 L175.4,89.7 L175.9,89.3 L176.3,89.0 L176.8,88.7
 L177.2,88.5 L177.7,88.2 L178.2,87.9 L178.6,87.7 L179.1,87.4 L179.5,87.2 L180.0,86.9 L180.4,86.6
 L180.9,86.4 L181.4,86.2 L181.8,86.0 L182.3,85.8 L182.7,85.6 L183.2,85.4 L183.7,85.3 L184.1,85.1
 L184.6,84.9 L185.0,84.7 L185.5,84.5 L185.9,84.4 L186.4,84.3 L186.9,84.1 L187.3,84.0 L187.8,83.9
 L188.2,83.8 L188.7,83.7 L189.2,83.6 L189.6,83.5 L190.1,83.4 L190.5,83.3 L191.0,83.3 L191.4,83.2
 L191.9,83.2 L192.4,83.1 L192.8,83.1 L193.3,83.0 L193.7,83.0 L194.2,82.9 L194.6,82.9 L195.1,82.9
 L195.6,82.9 L196.0,82.9 L196.5,82.9 L196.9,82.9 L197.4,82.9 L197.9,82.9 L198.3,82.9 L198.8,82.9
 L199.2,82.9 L199.7,82.9 L200.1,83.0 L200.6,83.0 L201.1,83.1 L201.5,83.1 L202.0,83.1 L202.4,83.2
 L202.9

## TABLE B4 - Benchmark quantities at distinct values in time for the finest solutions of the respective groups

### Minimum circularity and instance of time

In [33]:
public static (double circMin, double time) GetMinimumCircularity(Plot2Ddata.XYvalues circData) {
    double circMin = circData.Values[0];
    int minIndex = 0;
    for (int i = 1; i < circData.Values.Length; ++i) {
        if (circData.Values[i] < circMin) {
            circMin = circData.Values[i];
            minIndex = i;
        }
    }

    return (circMin, circData.Abscissas[minIndex]);
}

### Maximum rise velocity and instance of time

In [34]:
public static (double riseVelMax, double time) GetMaximumRiseVelocity(Plot2Ddata.XYvalues riseData) {
    double riseMax = riseData.Values[0];
    int maxIndex = 0;
    for (int i = 1; i < riseData.Values.Length; ++i) {
        if (riseData.Values[i] > riseMax) {
            riseMax = riseData.Values[i];
            maxIndex = i;
        }
    }

    return (riseMax, riseData.Abscissas[maxIndex]);
}

### Terminal rise height

In [35]:
public static double GetTerminalRiseHeight(Plot2Ddata.XYvalues yData) {
    return yData.Values.Last();
}

In [36]:
public static void CheckBenchmarkQuantities(List<Plot2Ddata> data, string study, int pDeg, int meshSize, double[] refValues) {

    int index = -1;

    // minimum circularity and instance of time
    {    
        string[] qNameHints = new string[] { "circularity" };
        for (int i = 0; i < data.Count(); i++) {
            if (qNameHints.Contains(data.ElementAt(i).Ylabel))
                index = i;
        }
        
        var datGroups = data.ElementAt(index).dataGroups;
        Plot2Ddata.XYvalues circData = null;
        for (int i = 0; i < datGroups.Count(); i++) {
            if (datGroups[i].Name.Contains($"{study}-ConvStudy-k{pDeg}-mesh{meshSize}")) {
                circData = datGroups[i];
            }
        }

        (double circMin, double time) = GetMinimumCircularity(circData);
        Console.WriteLine($"Minimum circularity: {circMin} at time {time}");
        // NUnit.Framework.Assert.AreEqual(refValues[0], circMin, 1e-4, 
        //     $"Minimum circularity does not match reference value {refValues[0]}.");
        // NUnit.Framework.Assert.AreEqual(refValues[1], time, 1e-4, 
        //     $"Time for minimum circularity does not match reference value {refValues[1]}.");
    }

    // Maximum rise velocity and instance of time
    {    
        string[] qNameHints = new string[] { "rise velocity" };
        for (int i = 0; i < data.Count(); i++) {
            if (qNameHints.Contains(data.ElementAt(i).Ylabel))
                index = i;
        }
        
        var datGroups = data.ElementAt(index).dataGroups;
        Plot2Ddata.XYvalues riseVelData = null;
        for (int i = 0; i < datGroups.Count(); i++) {
            if (datGroups[i].Name.Contains($"{study}-ConvStudy-k{pDeg}-mesh{meshSize}")) {
                riseVelData = datGroups[i];
            }
        }

        (double riseVelMax, double time) = GetMaximumRiseVelocity(riseVelData);
        Console.WriteLine($"Maximum rise velocity: {riseVelMax} at time {time}");
        // NUnit.Framework.Assert.AreEqual(refValues[2], riseVelMax, 1e-4, 
        //     $"Maximum rise velocity does not match reference value {refValues[2]}.");
        // NUnit.Framework.Assert.AreEqual(refValues[3], time, 1e-4, 
        //     $"Time for maximum rise velocity does not match reference value {refValues[3]}.");
    }

    // Terminal rise height
    {    
        string[] qNameHints = new string[] { "center of mass - y" };
        for (int i = 0; i < data.Count(); i++) {
            if (qNameHints.Contains(data.ElementAt(i).Ylabel))
                index = i;
        }
        
        var datGroups = data.ElementAt(index).dataGroups;
        Plot2Ddata.XYvalues heightData = null;
        for (int i = 0; i < datGroups.Count(); i++) {
            if (datGroups[i].Name.Contains($"{study}-ConvStudy-k{pDeg}-mesh{meshSize}")) {
                heightData = datGroups[i];
            }
        }

        double height = GetTerminalRiseHeight(heightData);
        Console.WriteLine($"Terminal rise height: {height}");
        // NUnit.Framework.Assert.AreEqual(refValues[4], height, 5e-4, 
        //     $"Terminal rise height does not match reference value {refValues[4]}.");
    }
}

In [43]:
CheckBenchmarkQuantities(evalData, "tc2", 2, 80, new double[] { 0.5093, 2.9850, 0.2500, 0.7290, 1.1300});

Error: System.NullReferenceException: Object reference not set to an instance of an object.
   at Submission#33.GetMinimumCircularity(XYvalues circData)
   at Submission#36.CheckBenchmarkQuantities(List`1 data, String study, Int32 pDeg, Int32 meshSize, Double[] refValues)
   at Submission#43.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)